# 6.2 功能调试：输出不一致时怎样逐步定位

本节用逐元素加法 `C = A + B` 演示：**从一个错误坐标出发，检查输入和中间结果，找到最早出现差异的位置，再修复代码。**

大写 `A`、`B`、`C` 表示GM（全局内存）中的输入和输出，小写 `a`、`b`、`c` 表示一个任务使用的UB缓冲区。

<p class="course-note"><small>线上环境没有昇腾950设备，可以对照代码和已有输出学习；自行运行时需准备对应环境。</small></p>

## 1. 先看错误现象

输入是两个长度为256的FP32向量，每个任务处理128个元素。用固定随机种子生成输入，并在CPU上计算参考结果：

```python
generator = torch.Generator().manual_seed(20260918)
a_cpu = torch.randint(-32, 33, (256,), generator=generator).float()
b_cpu = torch.randint(-32, 33, (256,), generator=generator).float()
expected = a_cpu + b_cpu
```

<p class="course-note"><small>本例使用可精确表示的小整数及其加法结果，因此采用零容差比较。</small></p>

调用待排查的已编译算子 `kernel`，找出不同元素的下标：

```python
actual = kernel(a_cpu.npu(), b_cpu.npu()).cpu()
wrong_indices = (actual != expected).nonzero().flatten()
print("不一致元素数：", wrong_indices.numel())
if wrong_indices.numel() > 0:
    i = int(wrong_indices[0])
    print("首个差异：", i, actual[i].item(), expected[i].item())
```

错误版本的实际运行记录为：

```text
错误版本：不一致元素数 = 252
首个差异：i=0, actual=-12, expected=-16
任务与局部位置：bx=0, local=0
```

先检查第一个差异 `C[0]`。它属于任务 `bx = 0 // 128 = 0`，任务内的位置为 `0 % 128 = 0`。**此时还不能确定错误来自搬运、计算还是写回。**

<p class="course-note"><small>错误程序在完整比较时触发断言；本页保留其真实日志，末尾的可执行单元运行修复版。</small></p>

## 2. 沿数据路径设置观察点

按照 **输入 → GM到UB的搬运 → Vector加法 → 写回GM** 的顺序，比较每一步的实际结果与应有结果。首次出现差异的位置，就是优先排查的范围。

![输出不一致时的逐步排查路径](images/06_02_copy_debug.png)

<p class="course-note"><small>图中表示数据处理与排查顺序，不表示硬件执行时间。A和B都要检查。</small></p>

## 3. 逐步比较输入、中间结果和输出

### 3.1 输入：确认设备收到的数据正确

核对输入形状、类型和参考公式后，将设备输入复制回CPU，与原始输入做完整比较：

```python
a_npu, b_npu = a_cpu.npu(), b_cpu.npu()
torch.testing.assert_close(a_npu.cpu(), a_cpu, rtol=0, atol=0)
torch.testing.assert_close(b_npu.cpu(), b_cpu, rtol=0, atol=0)
```

两个断言均未报错，输入检查记录为：

```text
输入检查：A、B 的设备数据与 CPU 输入完全一致
输入形状、类型： (256,) torch.float32
应读取的数据：A[0]=-13, B[0]=-3
```

**设备输入与原始输入一致。**

任务0的第一个元素应计算 `-13 + (-3) = -16`。接下来检查搬运。

### 3.2 搬运：分别检查UB中的a和b

在两次 `T.copy` 之后、进入 `T.SimdVF()` 之前打印UB数据：

```python
T.print(a, msg="A_after_copy")
T.print(b, msg="B_after_copy")
```

`T.print` 输出缓冲区中的值，`msg` 用来标记观察位置。找到任务0、局部位置0对应的日志：

```text
[AIV Block 0/2] msg='A_after_copy' BlockIdx=0: buffer=a, index=0, dtype=float32 value=-13.000000
[AIV Block 0/2] msg='B_after_copy' BlockIdx=0: buffer=b, index=0, dtype=float32 value=1.000000
```

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">观察对象</th>
<th style="text-align: left; vertical-align: top;">应有值</th>
<th style="text-align: left; vertical-align: top;">实际值</th>
<th style="text-align: left; vertical-align: top;">结论</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;"><code>a[0]</code>，应来自 <code>A[0]</code></td>
<td style="text-align: left; vertical-align: top;">-13</td>
<td style="text-align: left; vertical-align: top;">-13</td>
<td style="text-align: left; vertical-align: top;">此处一致</td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;"><code>b[0]</code>，应来自 <code>B[0]</code></td>
<td style="text-align: left; vertical-align: top;">-3</td>
<td style="text-align: left; vertical-align: top;">1</td>
<td style="text-align: left; vertical-align: top;">加法前已经不同</td>
</tr>
</tbody>
</table>

**首次观察到的差异出现在B搬入UB之后。**

设备上的原始B已经核对过，下一步应检查源切片、偏移、目标UB和搬运范围，尚不能直接认定是索引错误。

<p class="course-note"><small>多任务日志可能交错，要按 <code>BlockIdx</code> 和 <code>index</code> 查找。单点比较用于缩小范围，修复后仍需比较完整输出。</small></p>

### 3.3 计算：核对实际输入的和

目前应重点排查B从GM到UB的搬运。继续检查计算和写回，可以确认错误如何传到最终输出。在 `T.SimdVF()` 结束后、写回GM之前打印：

```python
T.print(c, msg="C_before_writeback")
```

对应日志为：

```text
[AIV Block 0/2] msg='C_before_writeback' BlockIdx=0: buffer=c, index=0, dtype=float32 value=-12.000000
```

`c[0] = -12` 等于实际输入的和 `-13 + 1`，**此处没有发现新的加法错误**。

若a、b正确而c不同，再检查寄存器读写位置、加法和掩码。

### 3.4 写回：比较UB结果与GM输出

写回语句为：

```python
T.copy(c, C[begin:begin + TILE])
```

任务0的 `begin = 0`，所以 `c[0]` 对应 `C[0]`。两者均为 `-12`，此处写回一致。若两者不同，应检查写回范围、索引和多任务覆盖；本例则应回到更早出现差异的B搬运。

## 4. 核对B的读取代码，再修复

错误版本中，A和B使用了不同的读取起点：

```python
begin = bx * TILE
source_begin = ((bx + 1) % 2) * TILE
T.copy(A[begin:begin + TILE], a)
T.copy(B[source_begin:source_begin + TILE], b)
```

将两个任务编号代入：

<table style="margin-left: 0; margin-right: auto; text-align: left;">
<thead>
<tr>
<th style="text-align: left; vertical-align: top;">任务 <code>bx</code></th>
<th style="text-align: left; vertical-align: top;">应读取的A、B范围</th>
<th style="text-align: left; vertical-align: top;">实际读取的B范围</th>
</tr>
</thead>
<tbody>
<tr>
<td style="text-align: left; vertical-align: top;">0</td>
<td style="text-align: left; vertical-align: top;"><code>[0:128]</code></td>
<td style="text-align: left; vertical-align: top;"><code>[128:256]</code></td>
</tr>
<tr>
<td style="text-align: left; vertical-align: top;">1</td>
<td style="text-align: left; vertical-align: top;"><code>[128:256]</code></td>
<td style="text-align: left; vertical-align: top;"><code>[0:128]</code></td>
</tr>
</tbody>
</table>

**B的两个数据块被交换了。**

任务0实际读取 `B[128] = 1`，与UB日志吻合。目标 `b` 和源切片均为128个FP32元素；这里需要修正的是源起点，让A、B读取同一范围：

```python
source_begin = begin
T.copy(B[source_begin:source_begin + TILE], b)
```

接下来用原输入重测，再换其他输入检查完整输出。

## 5. 运行修复后的完整程序

下面的完整程序保留修复处，并使用多组输入检查全部输出。

<p class="course-note"><small>在Notebook中运行时，以本章目录为工作目录，从下方准备单元开始顺序执行。<code>Writing</code>、<code>Overwriting</code>和<code>Appending</code>表示保存代码，最后的运行单元才执行NPU计算。</small></p>

In [1]:
from pathlib import Path
import re
import subprocess
import sys

Path("src").mkdir(exist_ok=True)
Path("answer").mkdir(exist_ok=True)

def run_and_show(command):
    # NPU程序在独立进程中运行；当前Notebook内核不导入torch_npu。
    result = subprocess.run(
        command, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    )
    # 只省略安装目录；数值、校验结果和报错内容保持原样。
    output = result.stdout.replace(str(Path.cwd()), ".")
    output = re.sub(r"/home/[^\s]+", "<已省略安装路径>", output)
    print(output, end="")
    result.check_returncode()
    return result


### 5.1 保存算子代码

修复处是 `source_begin = begin`，使A、B使用相同的读取起点。其余仍是前面学过的UB搬运、SIMD加法和GM写回。

<p class="course-note"><small>在编译前调用 <code>tilelang.disable_cache()</code>，关闭当前进程的TileLang编译缓存。</small></p>

In [2]:
%%writefile src/debug_fixed_demo.py
import tilelang
import tilelang.language as T
from tilelang.language import simd as S
import torch
import torch_npu

tilelang.disable_cache()

N = 256
TILE = 128

def make_add():
    @T.prim_func
    def add(
        A: T.Tensor((N,), "float32"),
        B: T.Tensor((N,), "float32"),
        C: T.Tensor((N,), "float32"),
    ):
        with T.Kernel(N // TILE) as bx:
            a = T.alloc_shared((TILE,), "float32")
            b = T.alloc_shared((TILE,), "float32")
            c = T.alloc_shared((TILE,), "float32")
            begin = bx * TILE
            source_begin = begin
            T.copy(A[begin:begin + TILE], a)
            T.copy(B[source_begin:source_begin + TILE], b)
            with T.SimdVF():
                full = S.pset(32, "PAT_ALL")
                for j in T.serial(TILE // 64):
                    va = S.vld(a[j * 64])
                    vb = S.vld(b[j * 64])
                    S.vsts(c[j * 64], S.vadd(va, vb, full), full)
            T.copy(c, C[begin:begin + TILE])
    return add



Overwriting src/debug_fixed_demo.py


### 5.2 添加完整输出比较

依次测试原来的复现输入、全零、常量和另一组随机输入，**每组都比较全部256个元素**。任何一组触发断言，程序都会停止。

In [3]:
%%writefile -a src/debug_fixed_demo.py

torch.npu.set_device(0)
kernel = tilelang.compile(make_add(), target="ascend", out_idx=-1)
generator = torch.Generator().manual_seed(20260918)
random_a = torch.randint(-32, 33, (N,), generator=generator).float()
random_b = torch.randint(-32, 33, (N,), generator=generator).float()
cases = [
    ("复现输入", random_a, random_b),
    ("全零输入", torch.zeros(N), torch.zeros(N)),
    ("常量输入", torch.full((N,), 2.0), torch.full((N,), -3.0)),
    ("另一组随机输入",
     torch.randint(-32, 33, (N,), generator=generator).float(),
     torch.randint(-32, 33, (N,), generator=generator).float()),
]
for name, a_cpu, b_cpu in cases:
    actual = kernel(a_cpu.npu(), b_cpu.npu()).cpu()
    expected = a_cpu + b_cpu
    torch.testing.assert_close(actual, expected, rtol=0, atol=0)
    print(f"{name}：全部 {N} 个元素通过比较")
print("Verification passed!")


Appending to src/debug_fixed_demo.py


### 5.3 执行并查看结果

在终端中运行：

```bash
python src/debug_fixed_demo.py
```

也可以执行下方Notebook单元。**四组输入全部通过后，才会打印最后的通过信息。**

In [4]:
_ = run_and_show([sys.executable, "-B", "src/debug_fixed_demo.py"])

Loading tilelang libs from dev root: <已省略安装路径>
[W918 19:45:40.931859719 FunctionLoader.cpp:48] Warning: LD_PRELOAD detected, FunctionLoader prefers RTLD_DEFAULT for symbol resolution. (function operator())
2026-09-18 19:45:52  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:131): TileLang begins to compile kernel `add` with `out_idx=-1`
2026-09-18 19:46:09  [TileLang:tilelang.jit.kernel:INFO] (kernel.py:139): TileLang completes to compile kernel `add`
复现输入：全部 256 个元素通过比较
全零输入：全部 256 个元素通过比较
常量输入：全部 256 个元素通过比较
另一组随机输入：全部 256 个元素通过比较
Verification passed!


## 6. 调试时的两个提示

**浮点容差：** GEMM等算子存在浮点累加，应根据数据类型和参考计算设置容差，不要直接沿用本例的零容差。

**编译报错：** 先根据报错检查接口、形状和存储作用域。设备代码尚未运行时，不会产生设备端打印。

<p class="course-note"><small>本例的<code>T.print</code>用于观察UB；其他存储作用域能否直接打印，需另行确认。</small></p>

---



[上一节：6.1 算子调试与性能测量](06.01_introduction.ipynb) · [下一节：6.3 性能测量](06.03_measure_and_tune.ipynb)